# Advertising Regression Analysis

Sıradan en küçük kareler yöntemi kullanılarak doğrusal regresyon modeli kurulumu ve eğitimi sonucunda yapılan tahminleri içerir.

###### Referans alınan kaynaklar:

* https://yigitsener.medium.com/makine-%C3%B6%C4%9Frenmesinde-python-ile-basit-do%C4%9Frusal-regresyon-modelinin-kurulmas%C4%B1-ve-yorumlanmas%C4%B1-4cf918e1adf
* https://pandas.pydata.org/docs/reference/
* https://numpy.org/doc/stable/reference/index.html#python-api
* https://matplotlib.org/stable/api/figure_api.html
* https://www.statsmodels.org/stable/api.html
* (2013). Pearson Education. Statistics for Business and Economics Eighth Edition. Paul Newbold, William L. Carlson, Betty M. Thorne.
* (2019). Cengage Learning. Statistics for Business & Economics 14e Metric Version. David R. Anderson. Dennis J. Sweeney. Thomas A. Williams. Jeffrey D. Camm. James J. Cochran. Michael J. Fry. Jeffrey W. Ohlmann.

In [ ]:
# gereken paketleri yükleyen import ediyoruz
import sys, subprocess
packages = ["pandas", "numpy", "statsmodels", "matplotlib"]
for pkg in packages:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])

import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt

# pandas kütüphanesi ile veriyi okur
df = pd.read_csv('Advertising.csv')

In [ ]:
df.head() # tablodaki veriyi tanımak ve görünür hata olmadığını anlamak için ilk birkaç satırı göster

In [ ]:
df.describe() # verilerin adedi, ortalaması, standart sapması gibi değerlerini göster

In [ ]:
df.corr() # verilerin korelasyon değerini gösterir

In [ ]:
"""
# sıradan en küçük kareler yöntemi ile tek bağımlı ve tek bağımsız değişkenle olacak şekilde eğitir;
# const:Sales bağımlı 
# var:TV bağımsız 

model_tv = sm.OLS(df['Sales'], sm.add_constant(df['TV'])).fit()
# regresyon modelinin özetini gösterir
print(model_tv.summary())
"""

In [ ]:
"""
# tek değişkenli modelleri eğitir ve özetleri toplar
independent_vars = ['TV', 'Radio', 'Newspaper']
rows = []

def rd(modNumber, decimals=4):
    return round(modNumber, decimals)

for var in independent_vars:
    mod = sm.OLS(df['Sales'], sm.add_constant(df[var])).fit()
    print(f"\n=== Model: Sales ~ {var} ===")
    print(mod.summary())

    ci = mod.conf_int()
    rows.append({
        'variable': var,
        'intercept': rd(mod.params['const']),
        'coef': rd(mod.params[var]),
        'std_err': rd(mod.bse[var]),
        't': rd(mod.tvalues[var]),
        'pvalue': rd(mod.pvalues[var]),
        'ci_lower': rd(ci.loc[var, 0]),
        'ci_upper': rd(ci.loc[var, 1]),
        'r_squared': rd(mod.rsquared),
        'adj_r_squared': rd(mod.rsquared_adj),
        'rmse': rd(np.sqrt((mod.resid ** 2).mean()))
    })

summary_df = pd.DataFrame(rows)
print(summary_df)
"""

In [ ]:
# ols modelini formula api'si kullanarak çoklu bağımsız değişkenlerle eğitir
# ~ const:Sales bağımlı
# + TV, Radio, Newspaper bağımsız
# data=df: referans alınan & veri çerçevesi
model = sm.formula.ols("Sales ~ TV + Radio + Newspaper", data=df).fit()
print(model.summary())

In [ ]:
sales_predicted_model = model.predict(df)
diff_bw = pd.DataFrame({'Gerçek Satışlar': df['Sales'], 'Tahmin Değerleri': sales_predicted_model}).sort_index()
diff_bw["Tahmin Hataları"] = diff_bw["Gerçek Satışlar"] - diff_bw["Tahmin Değerleri"]
diff_bw.to_excel('dist/diff_predicted.xlsx', index=False)
diff_bw